In [79]:
import pandas as pd
import os

# Create output directory if it doesn't exist
os.makedirs("data/exported", exist_ok=True)

In [80]:
# Import data [WILL NEED TO CHANGE BASED ON FORMATS FROM DATA GATHERING NOTEBOOK]
asylum_by_month = pd.read_csv("data/raw/asylum_cases_philadelphia_us.csv")
asylum_by_month['month'] = pd.to_datetime(asylum_by_month['fymon'])

asylum_by_representation = pd.read_csv("data/raw/asylum_cases_representation_philadelphia_us.csv")

# asylum_by_language = pd.read_excel("data/asylum.xlsx", sheet_name="Asylum by Language")
# asylum_by_language.columns = asylum_by_language.columns.str.lower().str.replace(" ", "_")

# asylum_by_custody = pd.read_excel("data/asylum.xlsx", sheet_name="Asylum by Custody")
# asylum_by_custody.columns = asylum_by_custody.columns.str.lower().str.replace(" ", "_")

In [81]:
# Helper function to estimate data on denied cases
def summarize(df, group_col=None):
    def agg(d):
        phl = d[d['city'] == 'Philadelphia']
        us  = d[d['city'] == 'Nationwide']

        phl_total  = phl['all_cases'].sum()
        phl_denied = phl['denied_cases'].sum()

        us_total   = us['all_cases'].sum()
        us_denied  = us['denied_cases'].sum()

        return pd.Series({
            'phl_total_cases':  phl_total,
            'phl_total_denied': phl_denied,
            'phl_pct_denied':   phl_denied / phl_total if phl_total else None,
            'us_total_cases':   us_total,
            'us_total_denied':  us_denied,
            'us_pct_denied':    us_denied / us_total if us_total else None,
        })

    if group_col:
        return df.groupby(group_col).apply(agg, include_groups=False).reset_index()
    return agg(df).to_frame().T

In [82]:
# Q1: Overall denial rates
asylum_total = summarize(asylum_by_month)
asylum_total.to_csv("data/exported/asylum_total.csv", index=False)
asylum_total

,phl_total_cases,phl_total_denied,phl_pct_denied,us_total_cases,us_total_denied,us_pct_denied
0,15498.0,9089.0,0.586463,989301.0,576210.0,0.582442


In [83]:
# Q2: By presidency
def assign_president(month):
    if pd.Timestamp("2001-01-20") < month <= pd.Timestamp("2009-01-20"):
        return "Bush"
    elif pd.Timestamp("2009-01-20") < month <= pd.Timestamp("2017-01-20"):
        return "Obama"
    elif pd.Timestamp("2017-01-20") < month <= pd.Timestamp("2021-01-20"):
        return "Trump I"
    elif pd.Timestamp("2021-01-20") < month <= pd.Timestamp("2025-01-20"):
        return "Biden"
    elif month > pd.Timestamp("2025-01-20"):
        return "Trump II"
    return "Other"

order = ["Bush", "Obama", "Trump I", "Biden", "Trump II"]
asylum_by_month['president'] = asylum_by_month['month'].apply(assign_president)
asylum_by_president = summarize(asylum_by_month, "president")
asylum_by_president['president'] = pd.Categorical(asylum_by_president['president'], categories=order, ordered=True)
asylum_by_president = asylum_by_president.sort_values('president')
asylum_by_president.to_csv("data/exported/asylum_by_president.csv", index=False)

asylum_by_president

,president,phl_total_cases,phl_total_denied,phl_pct_denied,us_total_cases,us_total_denied,us_pct_denied
1,Bush,5015.0,3466.0,0.691127,235377.0,135106.0,0.573998
2,Obama,1735.0,726.0,0.418444,167859.0,80639.0,0.480397
4,Trump I,2882.0,1710.0,0.593338,204651.0,135882.0,0.663969
0,Biden,3632.0,1629.0,0.448513,257952.0,130972.0,0.507738
5,Trump II,2005.0,1428.0,0.712219,115419.0,88939.0,0.770575
3,NaN,229.0,130.0,0.567686,8043.0,4672.0,0.580878


In [84]:
# Q3: Biden's last months vs. Trump II
recent = asylum_by_month[asylum_by_month['month'] >= pd.Timestamp("2024-04-01")].copy()
recent['president'] = recent['month'].apply(
    lambda m: "Biden" if m <= pd.Timestamp("2025-01-20") else ("Trump" if m >= pd.Timestamp("2025-02-01") else "Other")
)
asylum_biden_vs_trump = summarize(recent, "president")
asylum_biden_vs_trump['president'] = pd.Categorical(asylum_biden_vs_trump['president'], categories=["Biden", "Trump"], ordered=True)
asylum_biden_vs_trump = asylum_biden_vs_trump.sort_values('president')
asylum_biden_vs_trump.to_csv("data/exported/asylum_biden_vs_trump.csv", index=False)

asylum_biden_vs_trump

,president,phl_total_cases,phl_total_denied,phl_pct_denied,us_total_cases,us_total_denied,us_pct_denied
0,Biden,756.0,370.0,0.489418,70367.0,40198.0,0.571262
1,Trump,2005.0,1428.0,0.712219,115419.0,88939.0,0.770575


In [85]:
# Q4: By month (recent)
monthly = asylum_by_month[asylum_by_month['month'] >= pd.Timestamp("2024-07-01")].copy()
monthly['month'] = monthly['month'].dt.strftime('%Y-%m')
asylum_by_month_recent = summarize(monthly, "month")
asylum_by_month_recent.to_csv("data/exported/asylum_by_month_recent.csv", index=False)
asylum_by_month_recent

,month,phl_total_cases,phl_total_denied,phl_pct_denied,us_total_cases,us_total_denied,us_pct_denied
0,2024-07,48.0,26.0,0.541667,6959.0,3663.0,0.526369
1,2024-08,80.0,39.0,0.487500,6999.0,4021.0,0.574511
2,2024-09,68.0,35.0,0.514706,6462.0,3865.0,0.598112
3,2024-10,97.0,57.0,0.587629,7692.0,4578.0,0.595164
4,2024-11,92.0,46.0,0.500000,6639.0,4169.0,0.627956
5,2024-12,67.0,33.0,0.492537,6128.0,3762.0,0.613903
6,2025-01,83.0,53.0,0.638554,8348.0,5417.0,0.648898
7,2025-02,135.0,91.0,0.674074,10226.0,7350.0,0.718756
8,2025-03,154.0,91.0,0.590909,11908.0,8766.0,0.736144
9,2025-04,171.0,131.0,0.766082,12484.0,9446.0,0.756649


In [86]:
# # Q5: By custody status
# asylum_by_custody = summarise(asylum_by_custody, "custody")
# print(asylum_by_custody)
# asylum_by_custody.to_csv("data/exported/asylum_by_custody.csv", index=False)

In [87]:
# # Q6: By language
# asylum_by_language = summarise(asylum_by_language, "language")
# print(asylum_by_language)
# asylum_by_language.to_csv("data/exported/asylum_by_language.csv", index=False)

In [89]:
asylum_by_representation

,fymon,all_represented,city,all_not_represented,represented_denied,not_represented_denied
0,2000-10,48,Philadelphia,20,15,17
1,2000-11,24,Philadelphia,22,12,15
2,2000-12,29,Philadelphia,27,14,17
3,2001-01,37,Philadelphia,22,25,15
4,2001-02,31,Philadelphia,11,18,8
...,...,...,...,...,...,...
571,2025-08,8354,Nationwide,2017,6248,1633
572,2025-09,8570,Nationwide,2032,6804,1659
573,2025-10,8440,Nationwide,2176,6738,1877
574,2025-11,6373,Nationwide,1567,5184,1374


In [93]:
# Q7: By legal representation
# Modified version of "summarize" function, adapted to filenames
#asylum_by_representation.to_csv("data/exported/asylum_by_representation.csv", index=False)
d = asylum_by_representation
phl = d[d['city'] == 'Philadelphia']
us  = d[d['city'] == 'Nationwide']

phl_rep_total  = phl['all_represented'].sum()
phl_no_rep_total = phl['all_not_represented'].sum()
phl_no_rep_denied = phl['not_represented_denied'].sum()
phl_rep_denied = phl['represented_denied'].sum()

us_rep_total  = us['all_represented'].sum()
us_no_rep_total = us['all_not_represented'].sum()
us_no_rep_denied = us['not_represented_denied'].sum()
us_rep_denied = us['represented_denied'].sum()

series = pd.Series({
    'phl_rep_total': phl_rep_total,
    'phl_no_rep_total': phl_no_rep_total,
    'phl_no_rep_denied': phl_no_rep_denied,
    'phl_rep_denied': phl_rep_denied,
    'phl_pct_denied_with_rep': phl_rep_denied / phl_rep_total if phl_rep_total else None,
    'phl_pct_denied_without_rep': phl_no_rep_denied / phl_no_rep_total if phl_no_rep_total else None,
    'us_rep_total': us_rep_total,
    'us_no_rep_total': us_no_rep_total,
    'us_no_rep_denied': us_no_rep_denied,
    'us_rep_denied': us_rep_denied,
    'us_pct_denied_with_rep': us_rep_denied / us_rep_total if us_rep_total else None,
    'us_pct_denied_without_rep': us_no_rep_denied / us_no_rep_total if us_no_rep_total else None,
})

df = series.to_frame().T
df.to_csv("data/exported/asylum_by_representation.csv", index=False)
df[['phl_pct_denied_with_rep', 'phl_pct_denied_without_rep', 'us_pct_denied_with_rep', 'us_pct_denied_without_rep']]

,phl_pct_denied_with_rep,phl_pct_denied_without_rep,us_pct_denied_with_rep,us_pct_denied_without_rep
0,0.568994,0.749222,0.53052,0.79817
